# hftbacktest - TW Stock

Refactored notebook for Taiwan stock L2 backtest experiments.


## Setup

Set the symbol and time range, convert L2 data to hftbacktest events, then build shared backtest config.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.tw_stock_data_to_npz import convert_tw_stock_to_npz
from scripts.tw_stock_hftbacktest import BacktestConfig, import_hftbacktest
from scripts.tw_stock_strategies import (
    DEFAULT_QUEUE_MODELS,
    run_aggressive_fill_strategy,
    run_queue_model_comparison,
    run_strategy3_deep_queue_comparison,
    scan_strategy3_candidates,
)


In [ ]:
SYMBOL = "2308"
START_DATE = "2026-01-12"
END_DATE = "2026-01-15"
START_TIME = "12:00:00"
END_TIME = "13:30:00"

DATA_FILE, event_data = convert_tw_stock_to_npz(
    symbol=SYMBOL,
    start_date=START_DATE,
    end_date=END_DATE,
    start_time=START_TIME,
    end_time=END_TIME,
    workspace_root=ROOT,
)

hbtpkg = import_hftbacktest(ROOT)
CONFIG = BacktestConfig(data=DATA_FILE, order_latency_ns=0)
QUEUE_MODELS = DEFAULT_QUEUE_MODELS
DATA_FILE


## Strategy 1: Aggressive Fill at BBO

Buy at best ask, then sell at best bid. This should fill immediately by design.


In [ ]:
strategy1_output = run_aggressive_fill_strategy(
    CONFIG,
    hbtpkg,
    event_data,
    qty=1.0,
    round_trips=1,
)
strategy1_output


## Strategy 2: Passive Bid1/Ask1 Queue Model Comparison

Submit passive buy at bid1 and passive sell at ask1. Compare fill timestamps across queue models.


In [ ]:
strategy2_output, strategy2_fill_comparison = run_queue_model_comparison(
    CONFIG,
    hbtpkg,
    event_data,
    queue_models=QUEUE_MODELS,
    qty=1.0,
)
strategy2_fill_comparison[
    [
        "queue_model", "side", "order_id", "price", "exec_price", "exec_qty",
        "send_order_ts", "exch_ts", "local_ts", "fill_ts", "send_order_time",
        "exch_time", "local_time", "fill_time", "time_to_fill_s", "queue_model_fill_delta_ns",
        "step", "position", "balance", "equity",
    ]
]


## Strategy 3: Deeper Passive Orders on Cancel-Heavy Levels

Scan L2 events for same-price depth reductions with few opposite-side trades, then submit deeper passive orders with larger quantity and longer observation windows.


In [ ]:
strategy3_candidates = scan_strategy3_candidates(
    event_data,
    window_minutes=30,
    min_depth_reduce_events=20,
    max_trade_events=8,
    require_trade=True,
    limit=40,
)
strategy3_candidates[
    [
        "candidate_id", "window_start_time", "side", "px",
        "depth_reduce_events", "depth_reduce_qty", "trade_events", "trade_qty", "score",
    ]
].head(10)


In [ ]:
strategy3_output, strategy3_comparison = run_strategy3_deep_queue_comparison(
    CONFIG,
    hbtpkg,
    event_data,
    strategy3_candidates,
    queue_models=QUEUE_MODELS,
    qty=3.0,
    max_candidates=3,
    min_depth_level=2,
    max_window_s=6 * 60 * 60,
)
strategy3_comparison[
    [
        "candidate_id", "queue_model", "side", "price", "qty", "depth_level",
        "send_best_bid", "send_best_ask", "send_order_time", "fill_time", "was_filled",
        "time_to_fill_s", "queue_model_fill_delta_ns", "depth_reduce_events",
        "candidate_trade_events", "fill_step", "exec_price", "exec_qty",
    ]
]
